# ⚖️ Programmatic Evaluation Harness
**Series: 3/4** **Author:** Master Timo  
**Stack:** 100% Local (Ollama + JSON Constraints + `rich` console analytics)

### The Problem
Manual evaluation completely falls apart when scale increases. If you fine-tune a prompt template or tweak system parameters across dozens of user scenarios, manually reading every single response to check for compliance or omissions is impossible. Moving past "vibes-based prompt engineering" requires an objective, quantitative audit pipeline.

### The Engineering Solution
This notebook implements a local **LLM-as-a-Judge** framework. It passes structured scenario datasets through a high-capacity evaluator model (`gemma3` or `llama3.1`).
1. **Deterministic JSON Contracts:** Utilizing Ollama's native structure constraint API (`format: "json"`) alongside regex sanitation blocks to guarantee parse-safe dictionary outputs.
2. **Multi-Dimensional Metrics Scoring:** Grading system performance out of 5 across rigid, granular rubrics for *Completeness* and *Conciseness*.
3. **KPI Analytics Dashboard:** Running an aggregation loop that calculates total performance variance and slaps on a conditional operational binary status pass/fail marker at runtime.

### Pre-requisites
Ensure you pull a reasoning model suited for evaluator judging:
```bash
ollama pull gemma3:latest  # or llama3.1
pip install requests rich

### Code Implementation

In [1]:
%pip install requests rich

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
import requests
import json
import re
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track

# Initialize Rich Console for clean text UI layouts
console = Console()

OLLAMA_URL = "http://localhost:11434"
JUDGE_MODEL = "gemma3"
# 1. Check local background engine connection
try:
    health = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    models = [m["name"] for m in health.json().get("models", [])]
    
    print(f"✅ Ollama Local Engine is active.")
    print(f"📦 Available System Models: {models}")
    
    # Assert check for high-capacity reasoning judge
    if any(JUDGE_MODEL in m for m in models):
        print(f"🎯 Evaluator Judge Identity confirmed: '{JUDGE_MODEL}' tag is present.")
        
        # 🏎️ CRITICAL PERFORMANCE WARMUP: Pre-load weights into system memory context
        print("⏳ Warming up model weights inside memory layers...")
        warmup_payload = {"model": JUDGE_MODEL, "prompt": "Status check.", "stream": False}
        requests.post(f"{OLLAMA_URL}/api/generate", json=warmup_payload, timeout=120)
        print("🟢 Warmup complete. Core inference layer is operational.")
    else:
        print(f"⚠️  Performance Warning: '{JUDGE_MODEL}' missing. Run: ollama pull {JUDGE_MODEL}")
        
except Exception as e:
    print(f"❌ Ollama Environment Error: {e}\n👉 Please execute 'ollama serve' in your terminal daemon.")

    

✅ Ollama Local Engine is active.
📦 Available System Models: ['llama3.1:8b', 'llava:13b', 'nomic-embed-text:latest', 'gemma3:latest', 'mistral:latest']
🎯 Evaluator Judge Identity confirmed: 'gemma3' tag is present.
⏳ Warming up model weights inside memory layers...
🟢 Warmup complete. Core inference layer is operational.


In [13]:
# ---------------------------------------------------------
# ⚙️ CORE METRIC ENGINE
# ---------------------------------------------------------
def run_evaluation_judge(target_input: str, target_output: str, rubric: str) -> dict:
    """Passes target data to local judge model to receive strict numeric scoring JSON."""
    
    system_instruction = """You are an independent, objective QA evaluation system.
Your job is to audit output text against a rigid scoring rubric.
You must output your final verdict strictly as a raw JSON object matching the requested schema.

Expected Schema:
{
    "completeness_score": <integer value between 1 and 5>,
    "conciseness_score": <integer value between 1 and 5>,
    "critique": "<A brief one-sentence engineering justification for the marks assigned>"
}"""

    prompt = f"""
### TASK UNDER REVIEW
Input Given to System:
"{target_input}"

System Output to Evaluate:
"{target_output}"

### CRITERIA AND GRADING RUBRIC
{rubric}

Analyze the system response objectively. Return your metrics data JSON object:"""

    payload = {
        "model": JUDGE_MODEL,
        "prompt": prompt,
        "system": system_instruction,
        "stream": False,  
        "format": {
            "type": "object",
            "properties": {
                "completeness_score": {"type": "integer", "minimum": 1, "maximum": 5},
                "conciseness_score": {"type": "integer", "minimum": 1, "maximum": 5},
                "critique": {"type": "string"}
            },
            "required": ["completeness_score", "conciseness_score", "critique"]
        },
        "options": {
            "temperature": 0.0,
            "num_predict": 256  
        }
    }

    try:
        response = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=120)
        response.raise_for_status()
        raw_result = response.json()["response"].strip()
        
        if "```json" in raw_result:
            raw_result = re.search(r"```json\s*(\{.*?\})\s*```", raw_result, re.DOTALL).group(1)
            
        return json.loads(raw_result)
    except Exception as e:
        return {
            "completeness_score": 1,
            "conciseness_score": 1,
            "critique": f"Harness Execution Error: {str(e)}"
        }

In [14]:
def execute_eval_pipeline(dataset: list, rubric: str):
    """Executes evaluation across entire dataset and renders a rich analytics dashboard."""
    
    console.print(Panel(
        "[bold cyan]⚖️ Initializing Programmatic LLM-As-A-Judge Pipeline[/bold cyan]\nAuditing output dataset quality using local reasoning engines...", 
        border_style="cyan"
    ))
    
    scored_results = []
    
    for case in track(dataset, description="Processing Evaluation Suite..."):
        metrics = run_evaluation_judge(case["input"], case["output"], rubric)
        
        # Enforce type casting to guarantee clean mathematics calculation tracking arrays
        try:
            c_score = int(metrics.get("completeness_score", 1))
            n_score = int(metrics.get("conciseness_score", 1))
        except (ValueError, TypeError):
            c_score, n_score = 1, 1

        scored_results.append({
            "id": case["id"],
            "scenario": case["scenario"],
            "completeness": c_score,
            "conciseness": n_score,
            "critique": metrics.get("critique", "N/A")
        })
    
    total_cases = len(scored_results)
    avg_completeness = sum(r["completeness"] for r in scored_results) / total_cases
    avg_conciseness = sum(r["conciseness"] for r in scored_results) / total_cases
    
    metrics_table = Table(title="📈 System Prompt Performance Matrix", title_style="bold green")
    metrics_table.add_column("ID", justify="center", style="dim")
    metrics_table.add_column("Scenario Context", width=25)
    metrics_table.add_column("Completeness (1-5)", justify="center")
    metrics_table.add_column("Conciseness (1-5)", justify="center")
    metrics_table.add_column("Judge Audit Critique", width=55)
    
    for r in scored_results:
        comp_color = "green" if r["completeness"] >= 4 else "red" if r["completeness"] <= 2 else "yellow"
        conc_color = "green" if r["conciseness"] >= 4 else "red" if r["conciseness"] <= 2 else "yellow"
        
        metrics_table.add_row(
            str(r["id"]),
            r["scenario"],
            f"[{comp_color}]{r['completeness']}[/{comp_color}]",
            f"[{conc_color}]{r['conciseness']}[/{conc_color}]",
            f"[italic dim]{r['critique']}[/italic dim]"
        )
        
    console.print("\n")
    console.print(metrics_table)
    
    # 🟢 FIXED: Extracted text formatting boundaries into independent concatenation lines
    passed_compliance = avg_completeness >= 3.8 and avg_conciseness >= 3.8
    status_text = "[bold green]PASSING COMPLIANCE[/bold green]" if passed_compliance else "[bold red]FAILING COMPLIANCE[/bold red]"
    
    summary_text = (
        f"📊 [bold]EVALUATION RUN REPORT RECAP[/bold]\n"
        f"{"─" * 45}\n"
        f"🔹 Total Scenarios Checked: {total_cases}\n"
        f"🔹 Aggregated Completeness: [bold yellow]{avg_completeness:.2f} / 5.0[/bold yellow]\n"
        f"🔹 Aggregated Conciseness:  [bold yellow]{avg_conciseness:.2f} / 5.0[/bold yellow]\n"
        f"🔹 Operational Status: {status_text}"
    )
    
    console.print(Panel(summary_text, border_style="green" if passed_compliance else "red", width=55))

In [15]:
# ---------------------------------------------------------
# Production Evaluation Dataset Setup
# ---------------------------------------------------------
evaluation_rubric = """
1. Completeness (1-5): Does the response explicitly and accurately resolve the input query? Deduct points if core technical mitigation steps are hand-waved or missing.
2. Conciseness (1-5): Is it direct and free of conversational padding, marketing fluff, or generic greetings? High marks demand functional data density.
"""

production_eval_dataset = [
    {
        "id": 1,
        "scenario": "Database Connections",
        "input": "How do I fix a PostgreSQL 'Too many clients' error?",
        "output": "You need to modify your postgresql.conf file and increase the max_connections setting. Alternatively, you can spin up a connection pooler like PgBouncer to manage the connection overhead efficiently before traffic spikes."
    },
    {
        "id": 2,
        "scenario": "API Integration Auth",
        "input": "What HTTP status code should I return for an invalid API authentication payload?",
        "output": "Hello! Thanks for asking. When designing scalable APIs, dealing with security structures is highly vital. For an invalid auth scenario, you should almost universally look into returning a 401 Unauthorized payload code back to your client. Hope that helps you build your app!"
    },
    {
        "id": 3,
        "scenario": "Session Security Flags",
        "input": "Explain how to mitigate cross-site scripting (XSS) in session cookies.",
        "output": "To completely lock down your session variables against standard client-side script injection vectors, make sure you append both the HttpOnly and Secure parameter flags directly onto your Set-Cookie headers during the auth transaction pipeline."
    },
    {
        "id": 4,
        "scenario": "Memory Leak Overload",
        "input": "What causes an active Node.js server process to throw a continuous heap out-of-memory exception?",
        "output": "It could be due to a few things, honestly. Sometimes systems just get overloaded under heavy scale. Maybe try restarting the container configuration or upgrading your cloud cluster instances memory limit tiers to see if that keeps the server alive longer."
    }
]

execute_eval_pipeline(production_eval_dataset, evaluation_rubric)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ⚖️ Initializing Programmatic LLM-As-A-Judge Pipeline                                                             │
│ Auditing output dataset quality using local reasoning engines...                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

                                        📈 System Prompt Performance Matrix                                        
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃                           ┃ Complete… ┃ Concise… ┃                                                         ┃
┃ ID ┃ Scenario Context          ┃   (1-5)   ┃  (1-5)   ┃ Judge Audit Critique                                    ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1  │ Database Connections      │     4     │    4     │ The response provides two relevant solutions to the     │
│    │                           │           │          │ 'Too many clients' error, offering a direct             │
│    │                           │           │          │ configuration change and a more robust connection       │
│    │                           │           │          │ pooling solution, fulfilling the query's intent         │
│    │                           │           │          │ effectively and without unnecessary wording.            │
│ 2  │ API Integration Auth      │     2     │    2     │ The response identifies a relevant status code but      │
│    │                           │           │          │ includes unnecessary introductory and concluding        │
│    │                           │           │          │ remarks, and doesn't directly state the core answer     │
│    │                           │           │          │ without extraneous information.                         │
│ 3  │ Session Security Flags    │     5     │    5     │ The response directly addresses the query by outlining  │
│    │                           │           │          │ a key mitigation strategy for XSS attacks on session    │
│    │                           │           │          │ cookies – utilizing HttpOnly and Secure flags – and     │
│    │                           │           │          │ does so in a succinct manner.                           │
│ 4  │ Memory Leak Overload      │     2     │    3     │ The response identifies potential causes but lacks      │
│    │                           │           │          │ specific technical details about heap out-of-memory     │
│    │                           │           │          │ exceptions, offering only vague suggestions for         │
│    │                           │           │          │ remediation without diagnosing the root cause.          │
└────┴───────────────────────────┴───────────┴──────────┴─────────────────────────────────────────────────────────┘

╭─────────────────────────────────────────────────────╮
│ 📊 EVALUATION RUN REPORT RECAP                      │
│ ─────────────────────────────────────────────       │
│ 🔹 Total Scenarios Checked: 4                       │
│ 🔹 Aggregated Completeness: 3.25 / 5.0              │
│ 🔹 Aggregated Conciseness:  3.50 / 5.0              │
│ 🔹 Operational Status: FAILING COMPLIANCE           │
╰─────────────────────────────────────────────────────╯

## 🏁 Module 03 Summary Recap

| Evaluation Dimension | Metric Strategy | Execution Guardrail |
| :--- | :--- | :--- |
| **Completeness Audit** | Validates technical response accuracy against baseline queries. | 1-5 Quantitative Rubric |
| **Conciseness Audit** | Flags conversational fluff, introductory filler, and hedging. | 1-5 Quantitative Rubric |
| **Format Consistency** | Enforces structured data extraction outputs for data streams. | Native Ollama JSON Mode |